You can adapt it like this:

# Training and Fine-Tuning BERT for Sentiment Classification

## Classifying IMDB Movie Reviews from the Hugging Face Datasets Library

In this tutorial, we'll use the IMDb Movie Reviews Dataset dataset available through the Hugging Face Datasets library. This dataset contains 50,000 highly polarized movie reviews collected from IMDb, with an equal number of positive and negative reviews. It is one of the most widely used benchmark datasets for sentiment analysis in Natural Language Processing (NLP).

This notebook demonstrates how to train and fine-tune a BERT-based model for text classification using the popular Hugging Face `transformers` Python library.

We will fine-tune a **DistilBERT** model (`distilbert-base-uncased`) to classify movie reviews according to their sentiment. The task is a binary classification problem with the following labels:

* **NEGATIVE** (0)
* **POSITIVE** (1)

By the end of this tutorial, you will learn how to:

* Load and explore a text classification dataset using the Hugging Face Datasets library.
* Tokenize text data using a pretrained Transformer tokenizer.
* Fine-tune a pretrained DistilBERT model for sentiment classification.
* Evaluate model performance on unseen reviews.
* Use the trained model to predict sentiment for new movie reviews.

The IMDb dataset is particularly well-suited for learning Transformer-based text classification because it contains long-form reviews with rich sentiment expressions, making it an excellent benchmark for evaluating language understanding models.


**Basic steps involved in using BERT and HuggingFace:**
1. Divide your data into training and test sets.
2. Encode your data into a format BERT will understand.
3. Combine your data and labels into datset objects.
4. Load the pre-trained BERT model.
5. Fine-tune the model using your training data.
6. Predict new labels and evaluate performance on your test data.



<br><br>

## **Import necessary Python libraries and modules**

First, we will import necessary Python libraries and modules. These include as `gdown`, for downloading large files from Google Drive (where we will get our UCSD Goodreads reviews), as well as scikit-learn (`sklearn`) and PyTorch (`torch`), for various machine learning tools.

In [1]:
!pip3 install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 86.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
# Basic Python modules
from collections import defaultdict
import random
import pickle

# For downloading large files from Google Drive
# https://github.com/wkentaro/gdown
import gdown

# For working with gzip files
# https://docs.python.org/3/library/gzip.html
import gzip

# For working with JSON files
import json

# For data manipulation and analysis
import pandas as pd
import numpy as np

# For machine learning tools and evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# For deep learning
# https://pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html
import torch

# For plotting and data visualization
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import ticker
sns.set(style='ticks', font_scale=1.2)

The HuggingFace [`transformers` Python library](https://huggingface.co/transformers/installation.html) is included in Colab by default now, so we do not need to install it (but this is how you would install it with `pip`).

From `transformers`, we will import modules for `DistilBert`, a *distilled* or smaller version of a BERT model that runs more quickly and uses less computing power. This makes it ideal for those just getting started with BERT.

In [3]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments

<br><br>

## **Set parameters and file paths**

In [4]:
# This is the name of the BERT model that we want to use.
# We're using DistilBERT to save space (it's a distilled version of the full BERT model),
# and we're going to use the cased (vs uncased) version.
model_name = 'distilbert-base-uncased'

# This is the name of the program management system for NVIDIA GPUs. We're going to send our code here.
device_name = 'cuda'

# This is the maximum number of tokens in any document sent to BERT.
max_length = 512

# This is the name of the directory where we'll save our model. You can name it whatever you want.
cached_model_directory_name = 'distilbert-imdb-reviews'

<br><br>

## **Load and sample IMDB Reviews data**

In this tutorial we'll use the [Multilingual IMDB Reviews Corpus]  This is a large-scale collection of IMDB product reviews.

We can download the dataset from the Hugging Face Hub with the  Datasets library, but first let's take a look at the available subsets (also called configs):

In [5]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [6]:
for example in imdb['train']:
  print(example['text'])
  print(example['label'])
  break

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, eve

## Data Cleaning and Preprocessing

In [7]:
import re
import html

def clean_review(text):

    # Decode HTML entities
    text = html.unescape(text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

Next we loop through this dictionary and use `gdown` to download the Imdb review data.

# Model Params Defined

In [8]:
import pickle
import random

from pathlib import Path
from datasets import load_dataset
from transformers import DistilBertTokenizerFast, set_seed

PREPARED_DATA_PATH = "/kaggle/working/prepared_imdb.pkl"

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 512

TRAIN_SAMPLE_SIZE = 10000
TEST_SAMPLE_SIZE = 2000

SEED = 42

<br><br>

## **Split the data into training and test sets**

When training a machine learning model, it is necessary to split your training data into two parts: a "training" set and a "test" set.

We will train our BERT model on the "training" set of Goodreads reviews and then we will evaluate how well it is performing by running it on the "test" set of Goodreads reviews that the model has never seen before.

Normally, to tune the hyperparameters, you should also create a "validation" set for tuning, and only use the "test" set once, at the end of all tuning. For simplicity, in this tutorial, we will only using a training and test set.

In [9]:
def load_imdb_reviews(
    train_sample_size: int = None,
    test_sample_size: int = None,
):
    """
    Load IMDb reviews from Hugging Face.
    """

    imdb = load_dataset("stanfordnlp/imdb")

    train_reviews = list(imdb["train"])
    test_reviews = list(imdb["test"])

    if train_sample_size is not None:
        train_reviews = random.sample(
            train_reviews,
            min(train_sample_size, len(train_reviews))
        )

    if test_sample_size is not None:
        test_reviews = random.sample(
            test_reviews,
            min(test_sample_size, len(test_reviews))
        )

    return train_reviews, test_reviews


In [10]:
def build_label_maps():

    label2id = {
        "NEGATIVE": 0,
        "POSITIVE": 1
    }

    id2label = {
        0: "NEGATIVE",
        1: "POSITIVE"
    }

    return label2id, id2label

In [11]:
def prepare_data():

    set_seed(SEED)

    output_path = Path(PREPARED_DATA_PATH)

    train_reviews, test_reviews = load_imdb_reviews(
        train_sample_size=TRAIN_SAMPLE_SIZE,
        test_sample_size=TEST_SAMPLE_SIZE,
    )

    train_texts = [clean_review(row["text"]) for row in train_reviews] #preprocess train texts
    train_labels = [row["label"] for row in train_reviews]

    test_texts = [clean_review(row["text"]) for row in test_reviews] #preprocess test texts
    test_labels = [row["label"] for row in test_reviews]

    label2id, id2label = build_label_maps()

    tokenizer = DistilBertTokenizerFast.from_pretrained(
        MODEL_NAME
    )

    train_encodings = tokenizer(
        train_texts,
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    )

    test_encodings = tokenizer(
        test_texts,
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    )

    prepared = {
        "model_name": MODEL_NAME,
        "max_length": MAX_LENGTH,

        "train_texts": train_texts,
        "train_labels": train_labels,

        "test_texts": test_texts,
        "test_labels": test_labels,

        "label2id": label2id,
        "id2label": id2label,

        "train_encodings": dict(train_encodings),
        "test_encodings": dict(test_encodings),

        "train_labels_encoded": train_labels,
        "test_labels_encoded": test_labels,
    }

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with output_path.open("wb") as file:
        pickle.dump(prepared, file)

    print(f"Saved prepared data to {output_path}")
    print(f"Train examples: {len(train_texts)}")
    print(f"Test examples: {len(test_texts)}")

    return prepared

## Prepared data as per Model Format:

In [12]:
prepared = prepare_data()

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Saved prepared data to /kaggle/working/prepared_imdb.pkl
Train examples: 10000
Test examples: 2000


In [13]:
import json

with open('id2label.json','w') as fp:
    json.dump(prepared['id2label'], fp)

Here's an example of a training label and review:

In [14]:
train_texts = prepared['train_texts']
test_texts = prepared['test_texts']

In [15]:
train_texts[:2], prepared['train_labels'][:2]

(['Arguably this is a very good "sequel", better than the first live action film 101 Dalmatians. It has good dogs, good actors, good jokes and all right slapstick! Cruella DeVil, who has had some rather major therapy, is now a lover of dogs and very kind to them. Many, including Chloe Simon, owner of one of the dogs that Cruella once tried to kill, do not believe this. Others, like Kevin Shepherd (owner of 2nd Chance Dog Shelter) believe that she has changed. Meanwhile, Dipstick, with his mate, have given birth to three cute dalmatian puppies! Little Dipper, Domino and Oddball... Starring Eric Idle as Waddlesworth (the hilarious macaw), Glenn Close as Cruella herself and Gerard Depardieu as Le Pelt (another baddie, the name should give a clue), this is a good family film with excitement and lots more!! One downfall of this film is that is has a lot of painful slapstick, but not quite as excessive as the last film. This is also funnier than the last film. Enjoy "102 Dalmatians"! :-)',
 

<br><br>

## **Encode data for DistilBERT**

We're going to transform our texts and labels into a format that BERT (via Huggingface and PyTorch) will understand. This is called *encoding* the data.

Here are the steps we need to follow:

1. The labels&mdash;in this case, Goodreads genres&mdash;need to be turned into integers rather than strings.

2. The texts&mdash;in this case, Goodreads reviews&mdash;need to be truncated if they're more than 512 tokens or padded if they're fewer than 512 tokens. The tokens, or words in the texts, also need to be separated into "word pieces" and matched to their embedding vectors.

3. We need to add special tokens to help BERT:

| BERT special token | Explanation |
| --------------| ---------|
| [CLS] | Start token of every document. |
| [SEP] | Separator between each sentence |
| [PAD] | Padding at the end of the document as many times as necessary, up to 512 tokens |
|  &#35;&#35; | Start of a "word piece" |




Here we will load `DistilBertTokenizerFast` from the HuggingFace library, which will do all the work of encoding the texts for us. The `tokenizer()` will break word tokens into word pieces, truncate to 512 tokens, and add padding and special BERT tokens.

In [16]:
import json
import random
from pathlib import Path
from typing import Any

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score


def set_seed(seed: int) -> None:
    """Make sampling and model training more reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_label_maps(labels: list[str]) -> tuple[dict[str, int], dict[int, str]]:
    """Create stable label-to-id and id-to-label mappings."""
    unique_labels = sorted(set(labels))
    label2id = {label: index for index, label in enumerate(unique_labels)}
    id2label = {index: label for label, index in label2id.items()}
    return label2id, id2label


class ReviewDataset(torch.utils.data.Dataset):
    """PyTorch dataset wrapper for tokenized review texts and labels."""

    def __init__(self, encodings: dict[str, Any], labels: list[int]) -> None:
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        item = {key: torch.tensor(value[index]) for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[index])
        return item

    def __len__(self) -> int:
        return len(self.labels)


def compute_metrics(pred: Any) -> dict[str, float]:
    """Metrics callback used by HuggingFace Trainer."""
    labels = pred.label_ids
    predictions = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted", zero_division=0),
    }


In [17]:
import inspect
import os
import pickle
from pathlib import Path


PREPARED_DATA_PATH = "/kaggle/working/prepared_reviews.pkl"
MODEL_NAME = "distilbert-base-uncased"
MODEL_OUTPUT_DIR = "distilbert-imdb-reviews"
TRAINING_OUTPUT_DIR = "results/trainer"
LOGGING_DIR = "logs"
NUM_TRAIN_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 1e-5 #learning rate changed
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 50
EVAL_STRATEGY = "epoch"
SAVE_STRATEGY = "epoch"
SEED = 42
WANDB_PROJECT = "Mlops-Group-Project"
WANDB_RUN_NAME = "distilbert-run-v[2]"
HF_REPO_ID = "Nlp0187/distilbert-imdb-reviews-v2"


def load_prepared_data(path: str | Path) -> dict:
    with Path(path).open("rb") as file:
        return pickle.load(file)


def build_training_args():
    """Build TrainingArguments across transformers versions."""
    from transformers import TrainingArguments

    training_kwargs = {
        "output_dir": TRAINING_OUTPUT_DIR,
        "logging_dir": LOGGING_DIR,
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "per_device_train_batch_size": TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": EVAL_BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "warmup_steps": WARMUP_STEPS,
        "weight_decay": WEIGHT_DECAY,
        "logging_steps": LOGGING_STEPS,
        "save_strategy": SAVE_STRATEGY,
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1",
        "greater_is_better": True,
        "report_to": "wandb",
        "run_name": WANDB_RUN_NAME,
    }

    signature = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in signature.parameters:
        training_kwargs["eval_strategy"] = EVAL_STRATEGY
    else:
        training_kwargs["evaluation_strategy"] = EVAL_STRATEGY

    return TrainingArguments(**training_kwargs)



<br><br>

## **Set the BERT fine-tuning parameters**

These are the arguments we'll set in the HuggingFace TrainingArguments objects, which we'll then pass to the HuggingFace Trainer object. There are many more possible arguments, but here we highlight the basics and some common gotchas.

When training your own model, you should search over these parameters to find the best settings for your particular dataset. You should use a held-out set of validation data for this step.

| Parameter | Explanation |
|-----------| ------------|
| num_train_epochs | total number of training epochs (how many times to pass through the entire dataset; too much can cause overfitting) |
| per_device_train_batch_size | batch size per device during training |
| per_device_eval_batch_size |  batch size for evaluation |
|  warmup_steps |  number of warmup steps for learning rate scheduler (set lower because of small dataset size) |
| weight_decay | strength of weight decay (reduces size of weights, like regularization) |
| output_dir | output directory for the fine-tuned model and configuration files |
| logging_dir | directory for storing logs |
| logging_steps | how often to print logging output (so that we can stop training early if the loss isn't going down) |
| evaluation_strategy | evaluate while training so that we can see the accuracy going up |

In [18]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = user_secrets.get_secret("HF_TOKEN")
os.environ['WANDB_API_KEY'] = user_secrets.get_secret("WANDB_API_KEY")

<br><br>

## **Fine-tune the BERT model**

Then we create a HuggingFace `Trainer` object using the `TrainingArguments` object that we created above. We also send our `compute_metrics` function to the `Trainer` object, along with our test and train datasets.

**Note:** This is what we've been aiming for this whole time! All the work of tokenizing, creating datasets, and setting the training arguments was for this cell.

Time to finally fine-tune!

Be patient; if you've set everything in Colab to use GPUs, then it should only take a minute or two to run, but if you're running on CPU, it can take hours.

After every 10 steps (as we specified in the TrainingArguments object), the trainer will output the current state of the model, including the training loss, validation ("test") loss, and accuracy (from our `compute_metrics` function).

You should see the loss going down and the accuracy going up. If instead they are staying the same or oscillating, you probably need to change the fine-tuning parameters.

In [19]:
def train(prepared) -> Trainer:

    import os
    import wandb

    from transformers import (
        DistilBertForSequenceClassification,
        DistilBertTokenizerFast,
        Trainer,
    )

    set_seed(SEED)

    train_dataset = ReviewDataset(
        prepared["train_encodings"],
        prepared["train_labels_encoded"],
    )

    test_dataset = ReviewDataset(
        prepared["test_encodings"],
        prepared["test_labels_encoded"],
    )

    model = DistilBertForSequenceClassification.from_pretrained(
        prepared["model_name"],
        num_labels=len(prepared["label2id"]),
        label2id=prepared["label2id"],
        id2label=prepared["id2label"],
    )

    training_args = build_training_args()

    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        config={
            "dataset": "IMDb",
            "model": prepared["model_name"],
            "version" : "v2",
            "platform":"kaggle",
            "epochs": NUM_TRAIN_EPOCHS,
            "learning_rate": LEARNING_RATE,
            "batch_size": TRAIN_BATCH_SIZE,
            "max_length": prepared["max_length"],
            "train_examples": len(train_dataset),
            "test_examples": len(test_dataset),
        },
    )

    try:

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            compute_metrics=compute_metrics,
        )

        trainer.train()

        trainer.save_model(MODEL_OUTPUT_DIR)

        tokenizer = DistilBertTokenizerFast.from_pretrained(
            prepared["model_name"]
        )

        tokenizer.save_pretrained(
            MODEL_OUTPUT_DIR
        )

        hf_token = os.getenv("HF_TOKEN")
            
        if hf_token:

            from huggingface_hub import login

            login(token=hf_token)

            trainer.model.push_to_hub(
                HF_REPO_ID,
                private=False,
            )

            tokenizer.push_to_hub(
                HF_REPO_ID,
                private=False,
            )
            hf_model_url = f"https://huggingface.co/{HF_REPO_ID}"
            wandb.run.summary["huggingface_model"] = hf_model_url
            print(
                f"Uploaded model to https://huggingface.co/{HF_REPO_ID}"
            )

        else:

            print(
                "HF_TOKEN not found. Skipping upload."
            )

        print(
            f"Model saved to {MODEL_OUTPUT_DIR}"
        )

        return trainer

    finally:

        wandb.finish()

In [20]:
#train the model:
train(prepared)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Current

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.522732,0.459513,0.908000,0.907975
2,0.429644,0.455435,0.909000,0.908888
3,0.296224,0.455165,0.917000,0.916960


...

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

wandb: updating run metadata


Uploaded model to https://huggingface.co/Nlp0187/distilbert-imdb-reviews-v2
Model saved to distilbert-imdb-reviews


wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁▂█
wandb:                 eval/f1 ▁▂█
wandb:               eval/loss █▁▁
wandb:            eval/runtime █▁▁
wandb: eval/samples_per_second ▁██
wandb:   eval/steps_per_second ▁█▇
wandb:             train/epoch ▁▁▂▂▃▃▃▃▄▄▅▅▅▆▆▆▇▇▇███
wandb:       train/global_step ▁▁▂▂▃▃▃▃▄▄▅▅▅▆▆▆▇▇▇███
wandb:         train/grad_norm ▁▂▃▅▄▆▅▄█▅▂▅▃▆▆▄▄▆
wandb:     train/learning_rate ▄██▇▇▆▆▅▅▅▄▄▃▃▂▂▁▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.917
wandb:                 eval/f1 0.91696
wandb:               eval/loss 0.45516
wandb:            eval/runtime 17.2743
wandb: eval/samples_per_second 115.779
wandb:   eval/steps_per_second 1.852
wandb:       huggingface_model https://huggingface....
wandb:              total_flos 3974021959680000.0
wandb:             train/epoch 3
wandb:       train/global_step 939
wandb:                      +7 ...
wandb: 
wandb: 🚀 View run distilbert-run-v[2] at: htt

<br><br>

## **Model Inference**

In [21]:
import torch

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
)

HF_REPO_ID = "Nlp0187/distilbert-imdb-reviews-v2"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = DistilBertTokenizerFast.from_pretrained(
    HF_REPO_ID
)

model = DistilBertForSequenceClassification.from_pretrained(
    HF_REPO_ID
)

model.to(DEVICE)
model.eval()

tokenizer_config.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/784 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


But we might want to do more fine-grained analysis of the model, so we extract the predicted labels.

In [22]:
def predict(text: str):

    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True,
    )

    encoded = {
        k: v.to(DEVICE)
        for k, v in encoded.items()
    }

    with torch.no_grad():

        outputs = model(**encoded)

        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )

        predicted_id = torch.argmax(
            probabilities,
            dim=-1
        ).item()

        confidence = probabilities[
            0,
            predicted_id
        ].item()

    return {
        "label": model.config.id2label[predicted_id],
        "score": round(confidence, 4),
    }

In [23]:
review = """
This movie was fantastic.
The acting was brilliant and I loved every minute.
"""

print(predict(review))

{'label': 'POSITIVE', 'score': 0.9867}
